# CatBoost 二次优化版（v2）—— 时间分块CV + 删特征 + 轻调参

In [1]:

REF_DATE_STR = "2025-08-31"
TRAIN_CSV = "train.csv"
TEST_CSV  = "testaa.csv"
TRAIN_STM_FEAT = "train_statement_feature.csv"
TEST_STM_FEAT  = "testaa_statement_feature.csv"
OUT_DIR = "outputs"
N_FOLDS = 5
RANDOM_STATE = 42
USE_GPU = False

BASE_DROP = ['issue_time', 'record_time', 'history_time']
DROP_CANDIDATES = [
    [],
    ['diff_issue_record_days', 'diff_record_history_days'],
    ['diff_issue_record_days', 'diff_record_history_days', 'first_tx_days_ago', 'last_tx_days_ago'],
]

PARAM_CANDIDATES = [
    dict(iterations=5000, learning_rate=0.03, depth=6, l2_leaf_reg=9,  bootstrap_type='Bernoulli', subsample=0.75,
         rsm=0.8, random_strength=1.5, loss_function='Logloss', eval_metric='AUC', one_hot_max_size=10, verbose=False),
    dict(iterations=5000, learning_rate=0.03, depth=5, l2_leaf_reg=12, bootstrap_type='Bernoulli', subsample=0.70,
         rsm=0.8, random_strength=2.0, loss_function='Logloss', eval_metric='AUC', one_hot_max_size=10, verbose=False),
    dict(iterations=6000, learning_rate=0.025, depth=6, l2_leaf_reg=12, bootstrap_type='Bernoulli', subsample=0.70,
         rsm=0.8, random_strength=2.0, loss_function='Logloss', eval_metric='AUC', one_hot_max_size=10, verbose=False),
]


In [2]:

import os, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

os.makedirs(OUT_DIR, exist_ok=True)
REF_DATE = pd.Timestamp(REF_DATE_STR)

def _to_days_since_now(unix_series):
    dt = pd.to_datetime(unix_series, unit='s', utc=True, errors='coerce').dt.tz_convert(None)
    return (REF_DATE - dt).dt.days

def prepare_main_table(df):
    use_cols = [
        'id','title','career','zip_code','residence','loan','term','interest_rate',
        'issue_time','syndicated','installment','record_time','history_time',
        'total_accounts','balance_accounts','balance_limit','balance','level'
    ] + (['label'] if 'label' in df.columns else [])
    df = df[use_cols].copy()
    for c in ['issue_time','record_time','history_time']:
        df[f'{c}_days'] = _to_days_since_now(df[c])
    df['diff_issue_record_days']   = df['issue_time_days'] - df['record_time_days']
    df['diff_issue_history_days']  = df['issue_time_days'] - df['history_time_days']
    df['diff_record_history_days'] = df['record_time_days'] - df['history_time_days']
    df['utilization']    = (df['balance'] / df['balance_limit']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 10)
    df['accounts_ratio'] = (df['balance_accounts'] / df['total_accounts']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 1)
    def split_level(x):
        if isinstance(x, str) and len(x) >= 2: return x[0], x[1:]
        return 'NA', 'NA'
    lv = df['level'].fillna('NA')
    df['grade'], df['subgrade'] = zip(*lv.map(split_level))
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    for c in cat_cols:
        if pd.api.types.is_integer_dtype(df[c]):
            df[c] = df[c].astype('Int64').astype(str)
        else:
            df[c] = df[c].astype(str)
        df[c] = df[c].fillna('NA')
    return df

def merge_statement_feats(main_df, stm_path):
    if os.path.exists(stm_path):
        stm = pd.read_csv(stm_path)
        stm = stm[[c for c in stm.columns if c != 'label']].copy()
        main_df = main_df.merge(stm, on='id', how='left')
    else:
        main_df['stm_missing'] = 1
    num_exclude = ['id','label','title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    stm_num_cols = [c for c in main_df.columns if c not in num_exclude]
    main_df[stm_num_cols] = main_df[stm_num_cols].fillna(0)
    return main_df

def get_base_feature_sets(df):
    drop_cols = ['id','label']
    features = [c for c in df.columns if c not in drop_cols]
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    cat_cols = [c for c in cat_cols if c in features]
    return features, cat_cols

def make_time_forward_folds(df, time_col='issue_time_days', n_folds=5, purge=0.0):
    order = df[time_col].rank(method='first')
    bins = pd.qcut(order, n_folds, labels=False)
    folds = []
    for k in range(n_folds):
        valid_idx = df.index[bins == k].to_numpy()
        train_idx = df.index[bins < k].to_numpy()
        if purge > 0 and k > 0:
            valid_min = order[valid_idx].min()
            gap_mask = (order < valid_min) & (order >= valid_min - purge * (order.max() - order.min()))
            train_idx = df.index[(bins < k) & (~gap_mask)].to_numpy()
        folds.append((train_idx, valid_idx))
    return folds

def sanitize_params(params, use_gpu=False, y=None):
    p = dict(params)
    if p.get('bootstrap_type', '').lower() == 'bayesian' and 'subsample' in p:
        p.pop('subsample', None)
        p.setdefault('bagging_temperature', 1.0)
    if use_gpu:
        p['task_type'] = 'GPU'
    if y is not None:
        pos, neg = int(y.sum()), int(len(y) - y.sum())
        p['scale_pos_weight'] = float(neg / max(pos, 1))
    return p

def train_timecv_with_params(df_train, features, cat_cols, folds, params, early_stopping_rounds=300, use_gpu=False):
    X = df_train[features]
    y = df_train['label'].astype(int)
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]

    oof_pred = np.zeros(len(X))
    models, best_iters = [], []

    p = sanitize_params(params, use_gpu=use_gpu, y=y)

    for i, (tr_idx, va_idx) in enumerate(folds, 1):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
        train_pool = Pool(X_tr, label=y_tr, cat_features=cat_idx)
        valid_pool = Pool(X_va, label=y_va, cat_features=cat_idx)

        model = CatBoostClassifier(**p)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=200, early_stopping_rounds=early_stopping_rounds)
        oof_pred[va_idx] = model.predict_proba(valid_pool)[:,1]
        models.append(model)
        best_iters.append(model.tree_count_)

    oof_auc = roc_auc_score(y, oof_pred)
    return oof_auc, models, oof_pred, int(np.mean(best_iters))


In [3]:

# 读取与准备
tr = pd.read_csv(TRAIN_CSV)
tr = prepare_main_table(tr)
tr = merge_statement_feats(tr, TRAIN_STM_FEAT)

features, cat_cols = get_base_feature_sets(tr)
print("训练维度：", tr.shape)
print("初始特征数：", len(features))
print("类别特征：", cat_cols)


训练维度： (53480, 50)
初始特征数： 48
类别特征： ['title', 'career', 'zip_code', 'residence', 'term', 'syndicated', 'installment', 'level', 'grade', 'subgrade']


In [4]:

# 构造时间分块
time_folds = make_time_forward_folds(tr, time_col='issue_time_days', n_folds=N_FOLDS, purge=0.0)
print("Time-Forward CV 折数：", len(time_folds))


Time-Forward CV 折数： 5


In [5]:

# 小网格：删特征 + 轻调参
records = []
best_auc = -1.0
best_pack = None

for di, drop_extra in enumerate(DROP_CANDIDATES, 1):
    drop_list = BASE_DROP + drop_extra
    feats_try = [c for c in features if c not in drop_list]
    cats_try = [c for c in cat_cols if c in feats_try]
    if len(cats_try) == 0:
        print(f"[Skip] 无可用类别特征，drop={drop_list}")
        continue

    print(f"\n== Candidate DROP[{di}] -> 去掉 {len(drop_list)} 列：{drop_list}")
    for pi, params in enumerate(PARAM_CANDIDATES, 1):
        oof_auc, models, oof_pred, best_iters = train_timecv_with_params(
            tr, feats_try, cats_try, time_folds, params, early_stopping_rounds=300, use_gpu=USE_GPU
        )
        rec = dict(
            drop_id=di, param_id=pi, drop_cols="|".join(drop_list),
            oof_auc=round(oof_auc, 6), best_iters=best_iters,
            params=json.dumps(params, ensure_ascii=False)
        )
        records.append(rec)
        print(f"  -> Params[{pi}]  OOF(TimeCV)={oof_auc:.6f}  best_iters≈{best_iters}")

        if oof_auc > best_auc:
            best_auc = oof_auc
            best_pack = (feats_try, cats_try, params, models, oof_pred)

import pandas as pd, os
res_df = pd.DataFrame(records).sort_values('oof_auc', ascending=False)
res_path = os.path.join(OUT_DIR, "second_stage_results.csv")
res_df.to_csv(res_path, index=False, encoding='utf-8')
print(f"\n[SAVE] 搜索结果 -> {res_path}")
res_df.head(10)



== Candidate DROP[1] -> 去掉 3 列：['issue_time', 'record_time', 'history_time']


CatBoostError: Labels variable is empty.

In [ ]:

# 保存最优 OOF，并（可选）生成新提交
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from catboost import Pool

oof_v2_path = os.path.join(OUT_DIR, "v2_oof_predictions.csv")

if best_pack is not None:
    feats_best, cats_best, params_best, models_best, oof_best = best_pack
    pd.DataFrame({'id': tr['id'], 'label': tr['label'], 'oof_pred': oof_best}).to_csv(oof_v2_path, index=False, encoding='utf-8')
    print(f"[SAVE] 最优配置 OOF -> {oof_v2_path} | OOF(TimeCV)={roc_auc_score(tr['label'], oof_best):.6f}")

    if os.path.exists(TEST_CSV):
        te = pd.read_csv(TEST_CSV)
        te = prepare_main_table(te)
        te = merge_statement_feats(te, TEST_STM_FEAT)
        if 'label' in te.columns:
            te = te.drop(columns=['label'])
        pool = Pool(te[feats_best], cat_features=[te[feats_best].columns.get_loc(c) for c in cats_best])
        preds = np.mean([m.predict_proba(pool)[:,1] for m in models_best], axis=0)
        sub = pd.DataFrame({'id': te['id'], 'prob': preds})
        out_sub = os.path.join(OUT_DIR, 'test_pred_catboost_v2.csv')
        sub.to_csv(out_sub, index=False, encoding='utf-8')
        print(f"[SAVE] 新提交写入 -> {out_sub}")
else:
    print("[WARN] 未找到最优配置（records为空？请检查数据与参数）。")
